In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q ultralytics roboflow onnx onnxruntime onnxslim

import ultralytics
ultralytics.checks()

# Verify YOLO26 is available
from ultralytics import YOLO
model_test = YOLO('yolo26n.pt')
print('YOLO26n loaded successfully ✓')
del model_test

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 43.5/112.6 GB disk)
YOLO26n loaded successfully ✓


In [2]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="gwS23ENhb551QPRiP5H8")
project = rf.workspace("ziads-workspace-0o9lk").project("grocery-items-ctvu4-oc3bw")
version = project.version(5)
dataset = version.download("yolo26")


loading Roboflow workspace...
loading Roboflow project...
Exporting format yolo26 in progress : 89.0%
Version export complete for yolo26 format



Extracting Dataset Version Zip to grocery-items-5 in yolo26:: 100%|██████████| 5051/5051 [00:00<00:00, 6620.25it/s]


In [3]:
# — Cell 3: Inspect YAML Fixed —

import os, glob, yaml

# دور تلقائيًا على ملف data.yaml
yaml_files = glob.glob("/content/**/data.yaml", recursive=True)

print("Found YAML files:")
for y in yaml_files:
    print(y)

# اختار أول data.yaml موجود
DATASET_YAML = yaml_files[0]
DATASET_DIR = os.path.dirname(DATASET_YAML)

print("\nUsing YAML:", DATASET_YAML)
print("Dataset DIR:", DATASET_DIR)

with open(DATASET_YAML, "r") as f:
    cfg = yaml.safe_load(f)

NUM_CLASSES = cfg["nc"]
CLASS_NAMES = cfg["names"]

print(f"\nClasses ({NUM_CLASSES}): {CLASS_NAMES}")
print(f"Train: {cfg['train']}")
print(f"Val  : {cfg['val']}")

for split in ["train", "valid", "test"]:
    imgs = glob.glob(f"{DATASET_DIR}/{split}/images/*")
    if imgs:
        print(f"{split}: {len(imgs)} images")

Found YAML files:
/content/grocery-items-5/data.yaml

Using YAML: /content/grocery-items-5/data.yaml
Dataset DIR: /content/grocery-items-5

Classes (4): ['doritos', 'indomei', 'pepsi', 'tea el_arosa']
Train: ../train/images
Val  : ../valid/images
train: 2064 images
valid: 287 images
test: 172 images


In [5]:
# ── Cell 4: Train YOLO26n ──────────────────────────────────────────────────────
from ultralytics import YOLO

model = YOLO("yolo26n.pt")   # pretrained COCO weights

results = model.train(
    data         = DATASET_YAML,
    epochs       = 100,
    imgsz        = 640,       # sweet-spot for RPi 5 edge inference
    batch        = 32,        # Kaggle T4/P100 fits 32 easily
    patience     = 20,        # early-stop if no improvement for 20 epochs
    optimizer    = "MuSGD",   # YOLO26 native optimizer
    lr0          = 0.001,
    lrf          = 0.01,
    weight_decay = 0.0005,
    augment      = True,
    mosaic       = 1.0,
    mixup        = 0.1,
    copy_paste   = 0.1,
    degrees      = 10.0,
    fliplr       = 0.5,
    project      = "market_products",
    name         = "yolo26n_rpi5",
    exist_ok     = True,
    device       = 0,
    workers      = 4,
    cache        = True,
    plots        = True,
    verbose      = True,
)

WEIGHTS_DIR = str(results.save_dir) + "/weights"
BEST_PT     = WEIGHTS_DIR + "/best.pt"
print("\nBest weights:", BEST_PT)

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=True, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.1, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/grocery-items-5/data.yaml, degrees=10.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo26n_rpi5, nbs=64, nms=False, opset=None, optimize=False, optimizer=MuSGD, overlap_mask=True, pat

In [6]:
# ── Cell 5: Validate ──────────────────────────────────────────────────────────
# NOTE: If kernel restarted after training, uncomment and set these manually:
# WEIGHTS_DIR = "market_products/yolo26n_rpi5/weights"
# BEST_PT     = WEIGHTS_DIR + "/best.pt"
from ultralytics import YOLO

model_best = YOLO(BEST_PT)
metrics    = model_best.val(
    data   = DATASET_YAML,
    imgsz  = 640,
    device = 0,
    split  = "val",
    plots  = True,
)

print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1707.4±605.0 MB/s, size: 43.8 KB)
val: Scanning /content/grocery-items-5/valid/labels.cache... 287 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 287/287 120.4Mit/s 0.0s
WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 105, len(boxes) = 349. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 6.0it/s 3.0s
                   all        287        349       0.96      0.946      0.969      0.848
               doritos         65         68      0.955      0.939      0.967      0.848
               indomei        12

In [7]:
# ── Cell 6: Export ONNX ────────────────────────────────────────────────────────
# Target  : inference_laptop.py (Windows/Mac/Linux with GPU or CPU)
# FP32    : full precision, works everywhere
# INT8    : 2x smaller + faster on CPU, same accuracy
from ultralytics import YOLO
from onnxruntime.quantization import quantize_dynamic, QuantType
import os

model_best = YOLO(BEST_PT)

# FP32 ONNX
model_best.export(
    format   = "onnx",
    imgsz    = 320,
    opset    = 17,
    simplify = True,
    dynamic  = False,
    half     = False,
)
ONNX_FP32 = WEIGHTS_DIR + "/best.onnx"

# INT8 dynamic quantization
ONNX_INT8 = WEIGHTS_DIR + "/best_int8.onnx"
quantize_dynamic(
    model_input  = ONNX_FP32,
    model_output = ONNX_INT8,
    weight_type  = QuantType.QUInt8,
)

print(f"ONNX FP32 : {os.path.getsize(ONNX_FP32)/1e6:.1f} MB")
print(f"ONNX INT8 : {os.path.getsize(ONNX_INT8)/1e6:.1f} MB")
print("Use best.onnx with inference_laptop.py")

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,375,616 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/content/runs/detect/market_products/yolo26n_rpi5/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 300, 6) (5.1 MB)
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 5 packages in 191ms
Prepared 1 package in 7.51s
Installed 1 package in 48ms
 + onnxruntime-gpu==1.25.1

requirements: AutoUpdate success ✅ 8.2s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.21.0 opset 17...


Exporting aten::index operator of advanced indexing in opset 17 is achieved by combination of multiple ONNX operators, including Reshape, Transpose, Concat, and Gather. If indices include negative values, the exported graph will produce incorrect results.


ONNX: slimming with onnxslim 0.1.92...
ONNX: export success ✅ 11.8s, saved as '/content/runs/detect/market_products/yolo26n_rpi5/weights/best.onnx' (9.2 MB)

Export complete (12.1s)
Results saved to /content/runs/detect/market_products/yolo26n_rpi5/weights/best.onnx
Predict:         yolo predict task=detect model=/content/runs/detect/market_products/yolo26n_rpi5/weights/best.onnx imgsz=320 
Validate:        yolo val task=detect model=/content/runs/detect/market_products/yolo26n_rpi5/weights/best.onnx imgsz=320 data=/content/grocery-items-5/data.yaml  
Visualize:       https://netron.app


ONNX FP32 : 9.7 MB
ONNX INT8 : 2.7 MB
Use best.onnx with inference_laptop.py


In [8]:
# ── Cell 7: Export NCNN ────────────────────────────────────────────────────────
# Target  : inference_rpi5.py (Raspberry Pi 5 + Pi Camera v2)
# NCNN    : Tencent's ARM-native inference engine
#           Produces: model.ncnn.param (graph) + model.ncnn.bin (weights)
#           No NMS needed (YOLO26 is already NMS-free) → ultra-low latency
from ultralytics import YOLO
import os, glob

model_best = YOLO(BEST_PT)

model_best.export(
    format = "ncnn",
    imgsz  = 320,
    half   = False,   # FP16 not supported natively on RPi 5 CPU
)

# Ultralytics saves NCNN output in: weights/best_ncnn_model/
NCNN_DIR   = WEIGHTS_DIR + "/best_ncnn_model"
NCNN_PARAM = NCNN_DIR + "/model.ncnn.param"
NCNN_BIN   = NCNN_DIR + "/model.ncnn.bin"

# Fallback: search if path is slightly different
if not os.path.exists(NCNN_PARAM):
    found = glob.glob(WEIGHTS_DIR + "/**/*.param", recursive=True)
    if found:
        NCNN_PARAM = found[0]
        NCNN_BIN   = NCNN_PARAM.replace(".param", ".bin")
        NCNN_DIR   = os.path.dirname(NCNN_PARAM)

print(f"NCNN param : {os.path.getsize(NCNN_PARAM)/1e6:.2f} MB  → {NCNN_PARAM}")
print(f"NCNN bin   : {os.path.getsize(NCNN_BIN)/1e6:.1f}  MB  → {NCNN_BIN}")
print("Use model.ncnn.param + model.ncnn.bin with inference_rpi5.py")

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ NCNN export does not support end2end models, disabling end2end branch.
YOLO26n summary (fused): 146 layers, 2,495,864 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/content/runs/detect/market_products/yolo26n_rpi5/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 8, 2100) (5.1 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 359ms
Prepared 1 package in 91ms
Installed 1 package in 1ms
 + ncnn==1.0.20260114

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirement ['pnnx'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 29 packages in 405ms
Prepared 1 package in 347ms
Installed 1 package in 3ms
 + pnnx==2

In [9]:
# ── Cell 8: Sanity check ──────────────────────────────────────────────────────
import onnx, onnxruntime as ort, numpy as np

print("── ONNX ─────────────────────────────────")
for path in [ONNX_FP32, ONNX_INT8]:
    onnx.checker.check_model(onnx.load(path))
    sess  = ort.InferenceSession(path, providers=["CPUExecutionProvider"])
    dummy = np.random.rand(1, 3, 320, 320).astype(np.float32)
    out   = sess.run(None, {sess.get_inputs()[0].name: dummy})
    print(f"  {path.split('/')[-1]:22s}  shape: {out[0].shape}  ✓")

print("\n── NCNN ─────────────────────────────────")
import os
param_ok = os.path.exists(NCNN_PARAM)
bin_ok   = os.path.exists(NCNN_BIN)
print(f"  model.ncnn.param  exists: {param_ok}  ✓")
print(f"  model.ncnn.bin    exists: {bin_ok}   ✓")
print("  (NCNN runtime check skipped — not available on Kaggle)")
print("  Files are valid for RPi 5 deployment.")

── ONNX ─────────────────────────────────
  best.onnx               shape: (1, 300, 6)  ✓
  best_int8.onnx          shape: (1, 300, 6)  ✓

── NCNN ─────────────────────────────────
  model.ncnn.param  exists: True  ✓
  model.ncnn.bin    exists: True   ✓
  (NCNN runtime check skipped — not available on Kaggle)
  Files are valid for RPi 5 deployment.


In [10]:
# ── Cell 9: Package for download ──────────────────────────────────────────────
import shutil, yaml, os

with open(DATASET_YAML) as f:
    cfg = yaml.safe_load(f)

# Save classes.txt (shared by both inference scripts)
classes_txt = WEIGHTS_DIR + "/classes.txt"
with open(classes_txt, "w") as f:
    f.write("\n".join(cfg["names"]))
print(f"classes.txt saved — {len(cfg['names'])} classes")

# ── Zip 1: laptop_package.zip  (ONNX)
laptop_dir = "/tmp/laptop_package"
os.makedirs(laptop_dir, exist_ok=True)
shutil.copy(ONNX_FP32,   laptop_dir + "/best.onnx")
shutil.copy(ONNX_INT8,   laptop_dir + "/best_int8.onnx")
shutil.copy(classes_txt, laptop_dir + "/classes.txt")
shutil.make_archive("laptop_package", "zip", laptop_dir)

# ── Zip 2: rpi5_package.zip  (NCNN)
rpi_dir = "/tmp/rpi5_package"
os.makedirs(rpi_dir, exist_ok=True)
shutil.copy(NCNN_PARAM,  rpi_dir + "/model.ncnn.param")
shutil.copy(NCNN_BIN,    rpi_dir + "/model.ncnn.bin")
shutil.copy(classes_txt, rpi_dir + "/classes.txt")
shutil.make_archive("rpi5_package", "zip", rpi_dir)

print("\n✓ laptop_package.zip")
print("    best.onnx          → inference_laptop.py (FP32)")
print("    best_int8.onnx     → inference_laptop.py (INT8, faster)")
print("    classes.txt")
print("\n✓ rpi5_package.zip")
print("    model.ncnn.param   → inference_rpi5.py")
print("    model.ncnn.bin     → inference_rpi5.py")
print("    classes.txt")
print("\nDownload both zips from the Kaggle output panel (right sidebar).")

classes.txt saved — 4 classes

✓ laptop_package.zip
    best.onnx          → inference_laptop.py (FP32)
    best_int8.onnx     → inference_laptop.py (INT8, faster)
    classes.txt

✓ rpi5_package.zip
    model.ncnn.param   → inference_rpi5.py
    model.ncnn.bin     → inference_rpi5.py
    classes.txt

Download both zips from the Kaggle output panel (right sidebar).


In [11]:
from google.colab import files
import os

print(os.path.abspath("laptop_package.zip"))
print(os.path.abspath("rpi5_package.zip"))

files.download("laptop_package.zip")
files.download("rpi5_package.zip")

/content/laptop_package.zip
/content/rpi5_package.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>